# E1 — IPAD R01 Video Swin Tiny (Colab scaffold)

Bu notebook sequence sınırlarını koruyan 16-frame / stride-4 klipleri doğrular ve torchvision Video Swin Tiny modelini yüklemeye hazırlar. Eğitim hücreleri bilerek başlatılmaz; E1 pipeline ve frame timeline mapping ayrıca onaylanıp geliştirilecektir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT_ROOT='/content/drive/MyDrive/bitirme'
DATASET_ROOT='/content/drive/MyDrive/bitirme_dataset/R01'
CLIP_LENGTH=16
CLIP_STRIDE=4
BATCH_SIZE=2
NUM_WORKERS=2

In [ ]:
import torch
print('torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
 p=torch.cuda.get_device_properties(0); print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(p.total_memory/1024**3,1))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT_ROOT}/requirements.txt'], check=True)

In [ ]:
import sys, os
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0,PROJECT_ROOT); os.chdir(PROJECT_ROOT)
from src.training.train_frame_baseline import validate_manifest
from src.data.ipad_r01_dataset import R01FrameDataset
from src.models.frame_baseline import knn_scores
manifest_path=Path(PROJECT_ROOT)/'data/processed/r01_manifest.csv'
manifest=pd.read_csv(manifest_path)
if not manifest.path.map(lambda p:Path(p).is_file()).all():
 import subprocess
 subprocess.run([sys.executable,'scripts/prepare_r01.py','--root',DATASET_ROOT,'--output',str(manifest_path)],check=True)
 manifest=pd.read_csv(manifest_path)
train_df,val_df,_=validate_manifest(manifest); test_df=manifest[manifest.split=='test'].copy()
assert (train_df.label==0).all() and (val_df.label==0).all()
print('Sequence-level partitions:',train_df.sequence.nunique(),val_df.sequence.nunique(),test_df.sequence.nunique())

In [ ]:
# Sequence-local clip index: this function never crosses directories/sequences.
def build_sequence_clips(frame_paths, clip_length=CLIP_LENGTH, stride=CLIP_STRIDE):
    ordered=sorted(frame_paths,key=lambda p:int(Path(p).stem))
    return [[str(p) for p in ordered[i:i+clip_length]] for i in range(0,max(0,len(ordered)-clip_length+1),stride)]

clip_index={}
for split in ['train','val','test']:
    subset=manifest[manifest.split==split]
    for sequence,rows in subset.groupby('sequence',sort=True):
        clips=build_sequence_clips(rows.path.tolist())
        clip_index[(split,sequence)]=clips
print('Sequence-local clips:',sum(map(len,clip_index.values())))
print('Example clip length:',len(next(c for c in clip_index.values() if c)))
assert all(len(c)==CLIP_LENGTH for clips in clip_index.values() for c in clips)

In [ ]:
# Load the torchvision pretrained Video Swin Tiny backbone (optional model availability check)
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
weights=Swin3D_T_Weights.DEFAULT
video_model=swin3d_t(weights=weights)
video_model.head=torch.nn.Identity()
video_model=video_model.to(DEVICE).eval()
for p in video_model.parameters(): p.requires_grad=False
print('Video Swin Tiny loaded; output shape is checked with a dummy clip in the next cell.')

In [ ]:
# Smoke forward only: B,C,T,H,W input. No training is started.
with torch.inference_mode():
    dummy=torch.zeros(1,3,CLIP_LENGTH,224,224,device=DEVICE)
    feat=video_model(dummy)
print('Feature output:',tuple(feat.shape))

## E1 implementation checklist

- Use only normal train clips to build the memory bank; use held-out normal train sequences to calibrate threshold.
- Keep clip windows within each sequence (the index above does this).
- Read official test `.npy` labels only after threshold is fixed.
- Map each clip score onto its covered frame indices using a documented aggregation rule (for example overlap mean), then exact-merge on `split + sequence + frame_name`.
- Save config, threshold, predictions, metrics, checkpoints, and figures under a Drive run directory.

Bu notebook şu an klip/model iskeletini doğrular; E1 eğitimini başlatmaz.